# 7. Pace & outcomes: offensive-zone recoveries

Compare shot generation after **5v5 offensive-zone recoveries** by post-recovery total pace. Pool all datasets, use quartiles, and retain zero pace.

This is a retrospective association: measurement ends at the first shot for successes and possession end for failures.

## 7.1. Load inputs

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PROCESSED_DATA_DIR = Path("../data/processed")

# Use only owning-team recoveries that start a possession.
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values(
    "event_id"
)
movements = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
starts = events.groupby("possession_id", sort=False).head(1)
recoveries = starts.loc[
    starts["event"].eq("Puck Recovery")
    & starts["is_5v5"]
    & starts["event_team_zone"].eq("OZ")
    & starts["team_id"].eq(starts["possession_team_id"])
].copy()
possession_events = {
    possession_id: group
    for possession_id, group in events.loc[
        events["possession_id"].isin(recoveries["possession_id"])
    ].groupby("possession_id", sort=False)
}
assert all(group["is_5v5"].all() for group in possession_events.values())

## 7.2. Calculate post-recovery total pace

Pool movement through the first-shot approach, or possession end when no shot occurs.

In [2]:
def add_recovery_pace(recovery_events, transition_events, event_groups):
    movement_groups = {
        possession_id: group
        for possession_id, group in transition_events.groupby(
            "possession_id", sort=False
        )
    }
    rows = []
    for recovery in recovery_events.itertuples(index=False):
        possession = event_groups[recovery.possession_id]
        shots = possession.loc[
            possession["event_id"].gt(recovery.event_id)
            & possession["team_id"].eq(recovery.team_id)
            & possession["event"].isin(["Shot", "Goal"])
        ]
        first_shot_id = int(shots.iloc[0]["event_id"]) if not shots.empty else None
        cutoff_id = (
            first_shot_id if first_shot_id is not None else possession["event_id"].max()
        )

        # Include movement ending at the shot release, never subsequent flight.
        history = movement_groups.get(
            recovery.possession_id, transition_events.iloc[:0]
        )
        history = history.loc[
            history["start_event_id"].ge(recovery.event_id)
            & history["end_event_id"].le(cutoff_id)
        ]
        distance = float(history["distance_ft"].sum())
        duration = float(history["modeled_elapsed_seconds"].sum())
        rows.append(
            {
                "event_id": recovery.event_id,
                "first_shot_event_id": first_shot_id,
                "movement_transitions": len(history),
                "movement_distance_ft": distance,
                "movement_seconds": duration,
                "post_recovery_pace_ft_s": distance / duration
                if duration > 0
                else np.nan,
            }
        )
    totals = pd.DataFrame(
        rows,
        columns=[
            "event_id",
            "first_shot_event_id",
            "movement_transitions",
            "movement_distance_ft",
            "movement_seconds",
            "post_recovery_pace_ft_s",
        ],
    )
    totals["first_shot_event_id"] = totals["first_shot_event_id"].astype("Int64")
    return recovery_events.merge(totals, on="event_id", validate="one_to_one")


recoveries = add_recovery_pace(recoveries, movements, possession_events)

## 7.3. Identify shot outcomes

Success is a later recovering-team `Shot` or `Goal` in the same possession, including blocked and missed attempts.

In [3]:
recoveries["leads_to_shot"] = recoveries["first_shot_event_id"].notna()
first_shot_clock = recoveries["first_shot_event_id"].map(
    events.set_index("event_id")["modeled_clock_seconds"]
)
recoveries["time_to_first_shot_seconds"] = (
    recoveries["modeled_clock_seconds"] - first_shot_clock
)
assert recoveries["time_to_first_shot_seconds"].dropna().ge(0).all()

## 7.4. Summarize quartiles

In [4]:
def summarize_recoveries(data):
    eligible = data.loc[data["post_recovery_pace_ft_s"].notna()].copy()
    bands, edges = pd.qcut(
        eligible["post_recovery_pace_ft_s"],
        q=4,
        labels=False,
        retbins=True,
        duplicates="drop",
    )
    eligible["quartile"] = bands + 1
    boundaries = pd.DataFrame(
        {
            "quartile": np.arange(1, len(edges)),
            "lower_pace_ft_s": edges[:-1],
            "upper_pace_ft_s": edges[1:],
        }
    )
    summary = (
        eligible.loc[eligible["quartile"].notna()]
        .groupby("quartile", observed=True)
        .agg(
            possessions=("possession_id", "size"),
            shot_possessions=("leads_to_shot", "sum"),
            games=("game_id", "nunique"),
            median_pace_ft_s=("post_recovery_pace_ft_s", "median"),
            median_time_to_shot_seconds=("time_to_first_shot_seconds", "median"),
        )
        .reset_index()
    )
    summary["shot_pct"] = 100 * summary["shot_possessions"] / summary["possessions"]
    return summary.merge(boundaries, on="quartile", validate="one_to_one"), boundaries


recovery_summary, quartile_boundaries = summarize_recoveries(recoveries)
# Keep missing movement separate from genuine zero pace.
coverage_df = pd.DataFrame(
    [
        {
            "qualifying_possessions": len(recoveries),
            "shot_possessions": int(recoveries["leads_to_shot"].sum()),
            "no_movement": int(recoveries["movement_transitions"].eq(0).sum()),
            "zero_duration": int(
                (
                    recoveries["movement_transitions"].gt(0)
                    & recoveries["movement_seconds"].eq(0)
                ).sum()
            ),
            "missing_pace": int(recoveries["post_recovery_pace_ft_s"].isna().sum()),
            "zero_pace": int(recoveries["post_recovery_pace_ft_s"].eq(0).sum()),
            "charted": int(recovery_summary["possessions"].sum()),
        }
    ]
)
display(coverage_df)

,qualifying_possessions,shot_possessions,no_movement,zero_duration,missing_pace,zero_pace,charted
0,1538,639,196,249,445,25,1093


## 7.5. Display recovery outcomes

In [5]:
QUARTILE_COLORS = ["#bfdbfe", "#93c5fd", "#3b82f6", "#1d4ed8"]


def recovery_outcomes_figure(summary, boundaries):
    bars = summary.set_index("quartile").reindex(range(1, len(boundaries) + 1))
    # Format missing conditional timing explicitly in the hover.
    hover = bars[
        [
            "possessions",
            "shot_possessions",
            "games",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
            "median_pace_ft_s",
        ]
    ].copy()
    hover["shot_time"] = bars["median_time_to_shot_seconds"].map(
        lambda value: (
            f"{value:.1f} s" if pd.notna(value) else "No shot-producing possessions"
        )
    )
    figure = go.Figure(
        go.Bar(
            x=[f"Q{q}" for q in bars.index],
            y=bars["shot_pct"],
            marker_color=QUARTILE_COLORS[: len(boundaries)],
            showlegend=False,
            text=[
                f"{value:.1f}%" if pd.notna(value) else "" for value in bars["shot_pct"]
            ],
            textposition="outside",
            cliponaxis=False,
            customdata=hover.to_numpy(),
            hovertemplate=(
                "%{x}<br>Shot-producing possessions: %{y:.1f}%"
                "<br>Possessions: %{customdata[0]:.0f}"
                "<br>Shot-producing possessions: %{customdata[1]:.0f}"
                "<br>Games: %{customdata[2]:.0f}"
                "<br>Pace range: %{customdata[3]:.2f} to %{customdata[4]:.2f} ft/s"
                "<br>Median pace: %{customdata[5]:.1f} ft/s"
                "<br>Median time to first shot (shot-producing possessions only): "
                "%{customdata[6]}"
                "<extra></extra>"
            ),
        )
    )
    if summary.empty:
        note = "No data: no measurable pace with assigned bands."
    elif len(boundaries) < 4:
        note = f"Only {len(boundaries)} bands: ties prevent four quartiles."
    else:
        note = None
    if note:
        figure.add_annotation(
            text=note, x=0.5, y=-0.30, xref="paper", yref="paper", showarrow=False
        )
    figure.update_layout(
        title="Post-recovery pace and shot generation | 5v5 offensive-zone recoveries",
        template="plotly_white",
        height=500,
        margin=dict(t=100, b=100),
    )
    figure.update_xaxes(
        title_text="Post-recovery total pace quartile",
        categoryorder="array",
        categoryarray=["Q1", "Q2", "Q3", "Q4"],
    )
    figure.update_yaxes(range=[0, 100], title_text="Shot-producing possessions (%)")
    return figure


recovery_figure = recovery_outcomes_figure(recovery_summary, quartile_boundaries)
recovery_figure.show()

## 7.6. Export recovery outcomes

In [6]:
oz_recovery_outcomes_df = (
    recovery_summary[
        [
            "quartile",
            "possessions",
            "shot_possessions",
            "games",
            "median_pace_ft_s",
            "median_time_to_shot_seconds",
            "shot_pct",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
        ]
    ]
    .astype(
        {
            "quartile": "Int64",
            "possessions": "Int64",
            "shot_possessions": "Int64",
            "games": "Int64",
            "median_pace_ft_s": "Float64",
            "median_time_to_shot_seconds": "Float64",
            "shot_pct": "Float64",
            "lower_pace_ft_s": "Float64",
            "upper_pace_ft_s": "Float64",
        }
    )
    .sort_values("quartile")
    .reset_index(drop=True)
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
export_path = PROCESSED_DATA_DIR / "oz_recovery_outcomes.parquet"
oz_recovery_outcomes_df.to_parquet(
    export_path, engine="pyarrow", compression="snappy", index=False
)
saved = pd.read_parquet(export_path)
pd.testing.assert_frame_equal(saved, oz_recovery_outcomes_df, check_exact=True)
print(f"Exported {len(saved):,} rows and {len(saved.columns)} columns to {export_path}")
display(saved)

Exported 4 rows and 9 columns to ..\data\processed\oz_recovery_outcomes.parquet


,quartile,possessions,shot_possessions,games,median_pace_ft_s,median_time_to_shot_seconds,shot_pct,lower_pace_ft_s,upper_pace_ft_s
0,1,274,117,34,5.0,1.0,42.70073,0.0,9.144411
1,2,273,133,34,12.272938,4.0,48.717949,9.144411,14.903214
2,3,273,156,34,17.734148,4.0,57.142857,14.903214,20.809807
3,4,273,147,34,25.285954,3.0,53.846154,20.809807,181.914815
